##### Εισαγωγή notebook καθαρισμού δεδομένων από πίνακα products
- Σε αυτό το notebook θα επικεντρωθούμε στην ανάλυση των δεδομένων του 1ου datframe, "products" το οποίο περιέχει πληροφορίες για την εκάστοτε υπηρεσία. Ύστετα θα ενώσουμε τους 2 πίνακες για να έχουμε το τελικό Dataset. Εισάγουμε τα δεδομένα από το προηγούμενο notebook όπου και τα είχαμε κατεβάσει για αποφυγή πολυπλοκότητας και προχωράμε στην επεξεργασία. Η εισαγωγή γίνεται διαβάζοντας το parquet αρχείο που αποθηκεύσαμε στο προηγούμενο notebook, και μετατρέποντας το σε datframe μέσω της αντίστοιχης pandas εντολής
- Το περίεργο - δύσκολο κομμμάτι της ανάλυσης αυτής, είνα ότι ενώ ο πίνακας με τα δεδομένα product είναι πλήρως ανοιγμένος, το πλήθος των στηλών του είναι μεταβαλόμενο ανά υπηρεσία. Οπότε θα πρέπει να ακολουθηθεί κάποια τεχνική, προκειμένου ο κώδικας να γίνει γενικός και να τρέχει για όλες τις υπηρεσίες. (Ενδεικτικά στο ec2 ο products έχει 79 στήλες ίσως και παραπάνω, ενώ στο s3 έχει 25, και ανάλογα στους άλλους παρόχους)

In [115]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [116]:
df_terms_final = pd.read_parquet('cleaned_aws_terms.parquet')
df_products = pd.read_parquet('cleaned_aws_products.parquet')

print("Terms shape:", df_terms_final.shape)
print("Products shape:", df_products.shape)

df_terms_final.head()

Terms shape: (1898, 5)
Products shape: (2024, 22)


,sku,rateCode,description,unit,priceUSD
0,U3KHECER6QCVQZ6T,U3KHECER6QCVQZ6T.JRTCKXETXF.6YS6EN2CT7,$0.60 per hour for CreateVpnConnection in Euro...,Hrs,0.600
1,3GWW3MJ3JVTNDT23,3GWW3MJ3JVTNDT23.JRTCKXETXF.6YS6EN2CT7,USD0.055 per hour for TransitGateway-Hours:Tra...,hour,0.055
2,H24SU884XW7MB5WC,H24SU884XW7MB5WC.JRTCKXETXF.6YS6EN2CT7,$0.005 per In-use public IPv4 address per hour,Hrs,0.005
3,EHGME54GNN6GTD8Y,EHGME54GNN6GTD8Y.JRTCKXETXF.D7RZ6WCPVW,$0.01 from 0 to 1 PB - Per GB charge to access...,GB,0.010
4,CWACH8635EGQMD9G,CWACH8635EGQMD9G.JRTCKXETXF.6YS6EN2CT7,$0.008 per hour per IPv4 address in contiguous...,Hrs,0.008


In [117]:
print (f"Dimensions (rows,cols): {df_products.shape}")
df_products.head()

Dimensions (rows,cols): (2024, 22)


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.endpointType,attributes.usagetype,attributes.operation,attributes.regionCode,attributes.servicename,attributes.vpnType,attributes.group,attributes.groupDescription,attributes.attachmentType,attributes.trafficDirection,attributes.transferType,attributes.fromLocation,attributes.fromLocationType,attributes.toLocation,attributes.toLocationType,attributes.fromRegionCode,attributes.toRegionCode
0,U3KHECER6QCVQZ6T,Cloud Connectivity,AmazonVPC,Europe (Spain),AWS Region,IPsec,EUS2-VPN-large-Usage-Hours:ipsec.1,CreateVpnConnection,eu-south-2,Amazon Virtual Private Cloud,VPN Large (5 Gbps),None,None,None,None,None,None,None,None,None,None,None
1,3GWW3MJ3JVTNDT23,None,AmazonVPC,Israel (Tel Aviv),AWS Region,None,ILC1-TransitGateway-Hours,TransitGatewayPeering,il-central-1,Amazon Virtual Private Cloud,None,AWSTransitGateway,Hourly charge for Transit Gateway Peering Atta...,Transit Gateway,None,None,None,None,None,None,None,None
2,QNCR32XES4QEB4UB,VPC Peering,AmazonVPC,US West (Oregon),AWS Region,None,USW2-OdbPeering-AZ-In-Bytes,,us-west-2,Amazon Virtual Private Cloud,None,None,None,None,AZ-In-Bytes,None,None,None,None,None,None,None
3,H24SU884XW7MB5WC,None,AmazonVPC,EU (Stockholm),AWS Region,None,EUN1-PublicIPv4:InUseAddress,,eu-north-1,Amazon Virtual Private Cloud,None,VPCPublicIPv4Address,Hourly charge for In-use Public IPv4 Addresses,None,None,None,None,None,None,None,None,None
4,EHGME54GNN6GTD8Y,VpcEndpoint,AmazonVPC,US West (Oregon),AWS Region,Resource,USW2-VpcResource-ODB-Consumer-Bytes,VpcResourceConsumer,us-west-2,Amazon Virtual Private Cloud,None,VpcResources,Per GB charge to access ODB Network Resources,None,None,None,None,None,None,None,None,None


Ξεκινάει μια έρευνα σχετικά με τις στήλες

In [118]:
df_products['sku'].isna().sum()


np.int64(0)

In [119]:
df_products['productFamily'].unique()
# df_products["productFamily"].isna().any()
# df_products["productFamily"].isna().sum()



array(['Cloud Connectivity', None, 'VPC Peering', 'VpcEndpoint',
       'VPC Encryption Controls', 'VPC Route Server'], dtype=object)

In [120]:
df_products['attributes.servicecode'].unique()
df_products["attributes.servicecode"].isna().sum()


np.int64(0)

In [121]:
df_products['attributes.location'].unique()
df_products["attributes.location"].isna().any()
df_products["attributes.location"].isna().sum()


np.int64(208)

In [122]:
df_products["attributes.locationType"].unique()

# df_products["attributes.locationType"].isna().sum()


array(['AWS Region', None, 'AWS Local Zone'], dtype=object)

In [123]:
df_products["attributes.regionCode"].unique()


array(['eu-south-2', 'il-central-1', 'us-west-2', 'eu-north-1',
       'us-east-2', 'ca-west-1', 'ap-east-1', 'me-south-1',
       'ap-southeast-1', 'ap-southeast-2', None, 'ap-southeast-3',
       'ap-southeast-5', 'me-central-1', 'ap-southeast-4', 'us-west-1',
       'eu-north-1-cph-1', 'eu-west-3', 'eu-central-1', 'mx-central-1',
       'ap-southeast-7', 'eu-west-1', 'ap-southeast-6', 'us-east-1',
       'ap-northeast-2', 'ap-northeast-3', 'eu-south-1', 'af-south-1',
       'ap-south-2', 'ca-central-1', 'us-east-1-scl-1',
       'eu-north-1-hel-1', 'eu-central-2', 'us-gov-east-1',
       'us-east-1-qro-1', 'ap-east-2', 'us-west-2-den-1',
       'us-east-1-mci-1', 'ap-south-1', 'sa-east-1', 'us-gov-west-1',
       'ap-northeast-1', 'eu-west-2', 'ap-northeast-1-tpe-1',
       'ap-south-1-del-1', 'us-east-1-atl-1', 'us-east-1-phl-1',
       'us-west-2-lax-1', 'us-east-1-iah-1', 'us-east-1-nyc-1',
       'me-south-1-mct-1', 'eu-central-1-ham-1', 'us-east-1-mia-1',
       'us-east-1-bue-

In [124]:
target_columns = [
    'sku',
    'productFamily',
    'attributes.servicecode',
    'attributes.location',
    'attributes.locationType',
    'attributes.regionCode',
    'attributes.servicename'
]

# Βρίσκει ποια από τα παραπάνω ΔΕΝ υπάρχουν στο df
missing_cols = [col for col in target_columns if col not in df_products.columns]

if not missing_cols:
    print("All 7 cols exist.")
else:
    print("Missing cols from the 7 set:", missing_cols)

All 7 cols exist.
